# Azure Database — IMDB Sentiment Project

This notebook connects to the Azure SQL database, uploads movie reviews, runs them through the trained model, and stores the predictions back in the cloud.

**Run the cells in order, top to bottom.**

In [10]:
# Install the database connector (run once)
!pip install pyodbc

## 1. Connect to Azure SQL

In [11]:
import pyodbc

server   = "imdb-sql-gsatya-2026.database.windows.net"   # full server name
database = "imdbdb"
username = "imdbadmin"
password = "PASSWORD"                                 # your password

conn_str = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"UID={username};"
    f"PWD={password};"
    "Encrypt=yes;TrustServerCertificate=no;Connection Timeout=30;"
)

conn = pyodbc.connect(conn_str)
cursor = conn.cursor()
print("Connected to Azure SQL!")

Connected to Azure SQL!


## 2. Create the table

Holds the review text, the true sentiment, and (later) the model's prediction. Safe to re-run — it drops and recreates the table.

In [12]:
cursor.execute("""
IF OBJECT_ID('reviews', 'U') IS NOT NULL DROP TABLE reviews;
CREATE TABLE reviews (
    id INT IDENTITY(1,1) PRIMARY KEY,
    review_text NVARCHAR(MAX),
    sentiment NVARCHAR(10),
    predicted_sentiment NVARCHAR(10),
    prediction_confidence FLOAT
);
""")
conn.commit()
print("Table 'reviews' created with all columns!")

Table 'reviews' created with all columns!


## 3. Load the data and upload a sample

In [13]:
import pandas as pd

df = pd.read_csv("IMDB Dataset.csv")
print("CSV shape:", df.shape)
df.head()

CSV shape: (50000, 2)


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [14]:
# Upload a 2000-row sample to Azure
cursor.fast_executemany = True

sample = df.sample(2000, random_state=42)
rows = list(sample[["review", "sentiment"]].itertuples(index=False, name=None))

cursor.executemany(
    "INSERT INTO reviews (review_text, sentiment) VALUES (?, ?)",
    rows
)
conn.commit()
print(f"Uploaded {len(rows)} reviews to Azure!")

Uploaded 2000 reviews to Azure!


## 4. Verify the upload

In [15]:
cursor.execute("SELECT COUNT(*) FROM reviews")
print("Total reviews in Azure:", cursor.fetchone()[0])

# Count by sentiment
pd.read_sql("SELECT sentiment, COUNT(*) AS total FROM reviews GROUP BY sentiment", conn)

Total reviews in Azure: 2000


C:\Users\gsaty\AppData\Local\Temp\ipykernel_29812\1303922007.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql("SELECT sentiment, COUNT(*) AS total FROM reviews GROUP BY sentiment", conn)


,sentiment,total
0,negative,1024
1,positive,976


## 5. Load the trained model

In [16]:
import tensorflow as tf
from tensorflow import keras

model = keras.models.load_model("imdb_sentiment_model.keras")
print("Model loaded!")

Model loaded!


## 6. Run cloud data through the model (actual vs predicted)

In [17]:
# Read 10 reviews from the cloud
data = pd.read_sql("SELECT TOP 10 id, review_text, sentiment FROM reviews", conn)

# Clean <br /> tags the SAME way as training
texts = data["review_text"].str.replace(r"<br\s*/?>", " ", regex=True).values

# Predict
preds = model.predict(tf.constant(texts))
data["predicted"]  = ["positive" if p[0] > 0.5 else "negative" for p in preds]
data["confidence"] = [round(float(p[0]), 2) for p in preds]

data[["sentiment", "predicted", "confidence"]]

C:\Users\gsaty\AppData\Local\Temp\ipykernel_29812\1985455303.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql("SELECT TOP 10 id, review_text, sentiment FROM reviews", conn)


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 518ms/step


,sentiment,predicted,confidence
0,positive,negative,0.01
1,positive,positive,0.96
2,negative,negative,0.01
3,positive,positive,0.99
4,negative,negative,0.16
5,positive,positive,0.86
6,positive,positive,1.00
7,positive,positive,1.00
8,negative,negative,0.31
9,negative,negative,0.01


## 7. Predict all reviews and write results back to Azure

In [18]:
# Pull every review that has no prediction yet
data = pd.read_sql("SELECT id, review_text FROM reviews WHERE predicted_sentiment IS NULL", conn)
print(f"Rows to predict: {len(data)}")

texts = data["review_text"].str.replace(r"<br\s*/?>", " ", regex=True).values
preds = model.predict(tf.constant(texts))

updates = [("positive" if p[0] > 0.5 else "negative", round(float(p[0]), 2), int(rid))
           for rid, p in zip(data["id"].values, preds)]

cursor.fast_executemany = True
cursor.executemany(
    "UPDATE reviews SET predicted_sentiment = ?, prediction_confidence = ? WHERE id = ?",
    updates
)
conn.commit()
print(f"Updated {len(updates)} rows with predictions!")

C:\Users\gsaty\AppData\Local\Temp\ipykernel_29812\547490850.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql("SELECT id, review_text FROM reviews WHERE predicted_sentiment IS NULL", conn)


Rows to predict: 2000
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
Updated 2000 rows with predictions!


## 8. Let the database compute the model's accuracy

In [19]:
accuracy = pd.read_sql("""
SELECT
    COUNT(*) AS total_reviews,
    SUM(CASE WHEN sentiment = predicted_sentiment THEN 1 ELSE 0 END) AS correct,
    CAST(100.0 * SUM(CASE WHEN sentiment = predicted_sentiment THEN 1 ELSE 0 END) / COUNT(*) AS DECIMAL(5,2)) AS accuracy_pct
FROM reviews
WHERE predicted_sentiment IS NOT NULL
""", conn)

accuracy

C:\Users\gsaty\AppData\Local\Temp\ipykernel_29812\580841935.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  accuracy = pd.read_sql("""


,total_reviews,correct,accuracy_pct
0,2000,1747,87.35
